In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

# Harness-only conversion helpers: the mined fixture is stored in target-side
# form, while the oracle must receive the semantically equivalent pandas form.
def _to_pandas_fixture(value):
    if isinstance(value, pl.DataFrame):
        return value.to_pandas()
    if isinstance(value, pl.Series):
        return value.to_pandas()
    if isinstance(value, list):
        return [_to_pandas_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_pandas_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_pandas_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_pandas_fixture(item) for key, item in vars(value).items()
        })
    return value

def _to_polars_fixture(value):
    if isinstance(value, pd.DataFrame):
        return pl.from_pandas(value)
    if isinstance(value, pd.Series):
        return pl.from_pandas(value)
    if isinstance(value, list):
        return [_to_polars_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_polars_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_polars_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_polars_fixture(item) for key, item in vars(value).items()
        })
    return value


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- img2table_azure_concat_lazy_migration ---
FIX_IMG2TABLE_AZURE_CONCAT_LAZY_MIGRATION_LIST_DFS = [pl.DataFrame({"value":["Hello","World"],"page":[1,1],"x1":[10,110],"y1":[10,10],"x2":[100,200],"y2":[30,30],"conf":[99.5,98.2]}), pl.DataFrame({"value":["Foo"],"page":[2],"x1":[5],"y1":[50],"x2":[50],"y2":[65],"conf":[97.0]})]

# --- img2table_azure_list_dfs_migration ---
FIX_IMG2TABLE_AZURE_LIST_DFS_MIGRATION_LIST_DFS = [pl.DataFrame({"value":["Hello","World"],"page":[1,1],"x1":[10,110],"y1":[10,10],"x2":[100,200],"y2":[30,30],"conf":[99.5,98.2]}), pl.DataFrame({"value":["Foo"],"page":[2],"x1":[5],"y1":[50],"x2":[50],"y2":[65],"conf":[97.0]})]
FIX_IMG2TABLE_AZURE_LIST_DFS_MIGRATION_WORD_ELEMENTS = [{"value":"Hello","page":1,"x1":10,"y1":10,"x2":100,"y2":30,"conf":99.5}, {"value":"World","page":1,"x1":110,"y1":10,"x2":200,"y2":30,"conf":98.2}]

print("✅ Fixtures loaded")
OCRDataframe = SimpleNamespace  # mock for testing


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_img2table_azure_concat_lazy_migration(list_dfs):
    return OCRDataframe(df=pd.concat(list_dfs))
    return None

def before_img2table_azure_list_dfs_migration(list_dfs, word_elements):
    list_dfs.append(pd.DataFrame(word_elements))
    return None

_oracle_img2table_azure_concat_lazy_migration = before_img2table_azure_concat_lazy_migration
def before_img2table_azure_concat_lazy_migration(*args, **kwargs):
    return _oracle_img2table_azure_concat_lazy_migration(
        *[_to_pandas_fixture(value) for value in args],
        **{key: _to_pandas_fixture(value) for key, value in kwargs.items()},
    )


In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_img2table_azure_concat_lazy_migration(list_dfs):

    return OCRDataframe(df=pl.concat(list_dfs))

def gen_img2table_azure_list_dfs_migration(list_dfs, word_elements):
    list_dfs.append(pl.DataFrame(word_elements))
    return None

# ── Test harness type adapters ─────────────────────────────────────────────

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if hasattr(r, "df"):
        r = r.df
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: img2table_azure_concat_lazy_migration ===

# L1 smoke – generated
try:
    _r = gen_img2table_azure_concat_lazy_migration(FIX_IMG2TABLE_AZURE_CONCAT_LAZY_MIGRATION_LIST_DFS)
    print("✅ L1 smoke gen_img2table_azure_concat_lazy_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_img2table_azure_concat_lazy_migration: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_img2table_azure_concat_lazy_migration(FIX_IMG2TABLE_AZURE_CONCAT_LAZY_MIGRATION_LIST_DFS)
    print("✅ L1 smoke before_img2table_azure_concat_lazy_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_img2table_azure_concat_lazy_migration: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_img2table_azure_concat_lazy_migration(FIX_IMG2TABLE_AZURE_CONCAT_LAZY_MIGRATION_LIST_DFS)
    _rg = gen_img2table_azure_concat_lazy_migration(FIX_IMG2TABLE_AZURE_CONCAT_LAZY_MIGRATION_LIST_DFS)
    compare(_rb, _rg, "img2table_azure_concat_lazy_migration")
except Exception as _e:
    print(f"❌ L2 equivalence img2table_azure_concat_lazy_migration: setup error — {type(_e).__name__}: {_e}")

# L3 edge - concatenate one schema-bearing empty OCR chunk.
try:
    _empty_pd = pd.DataFrame({
        "value": pd.Series(dtype="object"),
        "page": pd.Series(dtype="int64"),
        "x1": pd.Series(dtype="int64"),
        "y1": pd.Series(dtype="int64"),
        "x2": pd.Series(dtype="int64"),
        "y2": pd.Series(dtype="int64"),
        "conf": pd.Series(dtype="float64"),
    })
    _empty_pl = pl.DataFrame(schema={
        "value": pl.String, "page": pl.Int64,
        "x1": pl.Int64, "y1": pl.Int64,
        "x2": pl.Int64, "y2": pl.Int64,
        "conf": pl.Float64,
    })
    _rb = before_img2table_azure_concat_lazy_migration([_empty_pd])
    _rg = gen_img2table_azure_concat_lazy_migration([_empty_pl])
    compare(
        _rb, _rg,
        "L3 edge img2table_azure_concat_lazy_migration empty chunk",
        check_row_order=True,
    )
except Exception as _e:
    print(f"❌ L3 edge img2table_azure_concat_lazy_migration empty chunk: {type(_e).__name__}: {_e}")
